# IncidentCommander RL — Kaggle shard 2 / 3

**Workload:** every 3rd task starting at index 1
(~127 of 381 scenarios). Trains a LoRA on Qwen2.5-1.5B-Instruct using a
local Qwen2.5-7B-Instruct critic, both loaded from Kaggle Models inputs.

**Required Kaggle inputs** (Add Data → Models, then Datasets):
1. Model: `qwen-lm/qwen-2.5` → variation `1.5b-instruct` (the actor)
2. Model: `qwen-lm/qwen-2.5` → variation `7b-instruct` (the critic)

**Required notebook settings** (right-hand sidebar):
- Accelerator: `GPU T4 x2` or `GPU P100`
- Persistence: `Files only` (so `/kaggle/working/` survives restarts)
- Internet: `On` (needed to clone the public GitHub repo)

**Output:** `/kaggle/working/adapter_kaggle2.zip` — download from the
sidebar after the run finishes. Combine all 3 with
`scripts/merge_lora_adapters.py` on your laptop.


## 1. GPU + path sanity

In [ ]:
import os, glob, subprocess, sys, json, pathlib

print('--- GPU ---')
subprocess.run(['nvidia-smi', '-L'], check=False)

actor_dirs  = glob.glob('/kaggle/input/qwen-2.5/transformers/1.5b-instruct/*')
critic_dirs = glob.glob('/kaggle/input/qwen-2.5/transformers/7b-instruct/*')
assert actor_dirs,  'Actor model not attached. Add `qwen-lm/qwen-2.5 1.5b-instruct` via Add Data > Models.'
assert critic_dirs, 'Critic model not attached. Add `qwen-lm/qwen-2.5 7b-instruct` via Add Data > Models.'
ACTOR_PATH  = sorted(actor_dirs)[-1]   # latest version
CRITIC_PATH = sorted(critic_dirs)[-1]
print('actor :', ACTOR_PATH)
print('critic:', CRITIC_PATH)

## 2. Install deps (Kaggle has torch/transformers preinstalled — we just pin compatible versions)

In [ ]:
%pip install -q \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "bitsandbytes==0.45.5" \
    "trl==0.12.1" \
    "huggingface_hub>=0.25,<1.0" \
    "pydantic>=2,<3" \
    "datasets" "sentencepiece" "protobuf" "safetensors"

## 3. Clone the repo (public GitHub)

In [ ]:
import os, subprocess, pathlib
WORK = '/kaggle/working/incident-commander'
if not pathlib.Path(WORK).exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/r1cksync/meta-rl-hack.git', WORK], check=True)
os.chdir(WORK)
print('cwd =', os.getcwd())

## 4. Configure run (shard, paths, env vars)

In [ ]:
import os

os.environ['INCIDENT_COMMANDER_MOCK'] = 'true'
os.environ['IC_ACTOR_MODEL']     = ACTOR_PATH
os.environ['IC_CRITIC_PROVIDER'] = 'local'        # 7B critic on the same GPU
os.environ['IC_CRITIC_MODEL']    = CRITIC_PATH
os.environ['IC_TASK_MODE']       = 'all'           # full 381 corpus
os.environ['IC_TASK_SHARDS']     = '3'
os.environ['IC_TASK_SHARD']      = '1'
os.environ['IC_TOTAL_UPDATES']   = '60'            # ~6h on T4 / P100
os.environ['IC_ROLLOUTS']        = '3'
os.environ['IC_MAX_STEPS']       = '12'
os.environ['IC_CKPT_EVERY']      = '15'
os.environ['IC_RUN_NAME']        = 'kaggle2'

# HF_TOKEN is only needed if you want intermediate-checkpoint upload to a
# HF model repo. Otherwise leave unset and grab the final adapter from
# /kaggle/working/. Use Kaggle Secrets > Add-ons to inject it safely.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN attached from Kaggle Secrets')
except Exception:
    print('HF_TOKEN not set — checkpoints stay local (download from sidebar).')

# Sanity: print actor + critic mode.
print('actor :', os.environ['IC_ACTOR_MODEL'])
print('critic:', os.environ['IC_CRITIC_MODEL'])
print('shard :', os.environ['IC_TASK_SHARD'], '/', os.environ['IC_TASK_SHARDS'])

## 5. Train

In [ ]:
# The training script runs to completion. tqdm progress + ETA are streamed
# to stdout. Kaggle truncates very long outputs — adapter checkpoints are
# always written to /kaggle/working/incident-commander/colab/logs/ regardless.
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/run_training.py'],
                         check=False)
print('exit code:', result.returncode)

## 6. Package outputs for download

In [ ]:
import shutil, glob, pathlib

LOGS = pathlib.Path('colab/logs')
finals = sorted(LOGS.glob('adapter_kaggle2_final'))
ckpts  = sorted(LOGS.glob('adapter_kaggle2_u*'))
keep   = (finals or ckpts)
assert keep, 'No adapter directories found — check the training cell output for errors.'
src = keep[-1]
print('packaging', src)

dst = pathlib.Path('/kaggle/working/adapter_kaggle2.zip')
shutil.make_archive(str(dst.with_suffix('')), 'zip', root_dir=src)
print('zipped to', dst, 'size:', dst.stat().st_size, 'bytes')

# Also copy the JSON training log for plotting on your laptop.
for j in glob.glob('colab/logs/training_kaggle2*.json'):
    shutil.copy(j, '/kaggle/working/')
print('files in /kaggle/working/:')
for f in sorted(pathlib.Path('/kaggle/working/').iterdir()):
    print(' ', f.name, f.stat().st_size)

## Done

Download `adapter_kaggle2.zip` from the **Output** tab on the
right.  Repeat for the other two shards (notebooks 2 and 3), then on your
laptop run:

```powershell
python scripts/merge_lora_adapters.py `
    --inputs ./adapter_kaggle1 ./adapter_kaggle2 ./adapter_kaggle3 `
    --output ./adapter_merged
```

The merged adapter loads with the standard `peft` API on top of
`Qwen/Qwen2.5-1.5B-Instruct`.
